<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import re
import pickle
from collections import Counter
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load and merge data files
file_path_1 = 'data/raw/Dataset 1/train_login_form.csv'
file_path_2 = 'data/raw/Dataset 1/train_no_form.csv'

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)
df = pd.concat([df1, df2], ignore_index=True)
print(f"Total records loaded: {len(df)} (file1: {len(df1)}, file2: {len(df2)})")

# 1-1. Handle missing values
n_missing = df['html_signature'].isna().sum()
if n_missing:
    print(f"⚠️ Found {n_missing} missing html_signature values → replacing with empty string")
df['html_signature'] = df['html_signature'].fillna('')

# 1-2. Remove duplicate sha256 (in case the same page appears in both files)
before = len(df)
df = df.drop_duplicates(subset='sha256', keep='first').reset_index(drop=True)
if before != len(df):
    print(f"⚠️ Removed {before - len(df)} duplicate sha256 records")

# 1-3. Check class distribution
print("\nLabel distribution:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True).round(3))

# 2. Extract structural numeric features
def extract_structural_features(sig: str) -> dict:
    max_depth = 0
    curr_depth = 0
    for char in sig:
        if char == '(':
            curr_depth += 1
            max_depth = max(max_depth, curr_depth)
        elif char == ')':
            curr_depth = max(0, curr_depth - 1)  # guard against unbalanced parentheses

    tags = re.findall(r'[a-zA-Z0-9]+', sig)
    tag_counts = Counter(tags)
    form_count = tag_counts.get("form", 0)
    input_count = tag_counts.get("input", 0)

    return {
        "signature_length": len(sig),
        "max_depth": max_depth,
        "total_tags": len(tags),
        "unique_tags_count": len(tag_counts),
        "form_count": form_count,
        "input_count": input_count,
        "button_count": tag_counts.get("button", 0),
        "div_count": tag_counts.get("div", 0),
        "script_count": tag_counts.get("script", 0),
        "a_count": tag_counts.get("a", 0),
        "img_count": tag_counts.get("img", 0),
        "input_per_form_ratio": input_count / form_count if form_count > 0 else 0.0,
    }

df_struct = pd.DataFrame([extract_structural_features(sig) for sig in df['html_signature']])

# 3. Extract N-gram features (keep sparse)
cleaned_signatures = [re.sub(r'[\(\)]+', ' ', sig).strip() for sig in df['html_signature']]
vectorizer = CountVectorizer(ngram_range=(2, 3), token_pattern=r'\b\w+\b', min_df=2)
ngram_matrix = vectorizer.fit_transform(cleaned_signatures)  # keep as sparse matrix

# Save the vectorizer so it can be reused (transform) on test data later
with open('ngram_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

ngram_cols = [f"ngram_{name}" for name in vectorizer.get_feature_names_out()]

# 4. Final combination — keep structural features dense, n-gram sparse, merge later at model input time
meta_cols = df[['url', 'sha256', 'label']].reset_index(drop=True)
struct_df = df_struct.reset_index(drop=True)

X_preprocessed = pd.concat([meta_cols, struct_df], axis=1)
X_ngram_sparse = ngram_matrix  # scipy.sparse.csr_matrix, combine with sparse.hstack when training

print(f"\nStructural feature shape: {struct_df.shape}")
print(f"N-gram feature shape (sparse): {ngram_matrix.shape}")

# =========================================================
# 📊 [Output] Key feature statistics and top N-gram patterns
# =========================================================
print("\n" + "="*50)
print("1. Overall Statistics for Structural Numeric Features")
print("="*50)
struct_cols = ['max_depth', 'total_tags', 'form_count', 'input_count', 'button_count', 'script_count']
print(df_struct[struct_cols].describe().round(2))

if 'label' in df.columns and df['label'].nunique() > 1:
    print("\n" + "="*50)
    print("2. Class-wise Comparison of Structural Feature Means")
    print("="*50)
    comparison_df = pd.concat([df['label'], df_struct[struct_cols]], axis=1)
    print(comparison_df.groupby('label').mean().round(2))

print("\n" + "="*50)
print("3. Top 10 Most Frequent N-gram Patterns")
print("="*50)
ngram_sums = pd.Series(ngram_matrix.sum(axis=0).A1, index=ngram_cols)
top_ngrams = ngram_sums.sort_values(ascending=False).head(10)
for idx, (ngram_name, count) in enumerate(top_ngrams.items(), 1):
    clean_name = ngram_name.replace('ngram_', '')
    print(f"{idx:2d}. Tag pattern [{clean_name}]: appeared {int(count)} times")

Total records loaded: 1183 (file1: 502, file2: 681)
⚠️ Found 2 missing html_signature values → replacing with empty string

Label distribution:
label
NO_FORM                 681
LOGIN_FORM_MALICIOUS    502
Name: count, dtype: int64
label
NO_FORM                 0.576
LOGIN_FORM_MALICIOUS    0.424
Name: proportion, dtype: float64

Structural feature shape: (1183, 12)
N-gram feature shape (sparse): (1183, 5487)

1. Overall Statistics for Structural Numeric Features
       max_depth  total_tags  form_count  input_count  button_count  \
count    1183.00     1183.00     1183.00      1183.00       1183.00   
mean       16.94      501.25        1.24         8.11          7.10   
std         7.70      972.35        2.15        20.35         30.13   
min         0.00        0.00        0.00         0.00          0.00   
25%        11.00       86.00        0.00         1.00          0.00   
50%        17.00      127.00        1.00         7.00          1.00   
75%        22.00      550.00       

In [3]:
!git clone https://github.com/Smyles019/html-login-form-detector.git

Cloning into 'html-login-form-detector'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 28 (delta 2), reused 16 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 348.86 KiB | 4.78 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [4]:
%cd REPOSITORY

[Errno 2] No such file or directory: 'REPOSITORY'
/content


In [6]:
%cd html-login-form-detector

/content/html-login-form-detector


In [7]:
!git branch -a

* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main


In [8]:
!git checkout -b feature/ml-model

Switched to a new branch 'feature/ml-model'


In [9]:
!git branch


* feature/ml-model
  main


In [10]:
!ls


data  notebooks  src


In [34]:
import pandas as pd
import re
import pickle
from collections import Counter

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



# 1. LOAD AND MERGE DATA


file_path_1 = 'data/raw/Dataset 1/train_login_form.csv'
file_path_2 = 'data/raw/Dataset 1/train_no_form.csv'

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)

df = pd.concat([df1, df2], ignore_index=True)

print(f"Total records loaded: {len(df)}")
print(f"File 1 records: {len(df1)}")
print(f"File 2 records: {len(df2)}")



# 2. HANDLE MISSING VALUES


n_missing = df['html_signature'].isna().sum()

if n_missing > 0:
    print(
        f"Found {n_missing} missing html_signature values "
        f"→ replacing with empty strings"
    )

df['html_signature'] = df['html_signature'].fillna('')



# 3. REMOVE DUPLICATE PAGES


before = len(df)

df = (
    df.drop_duplicates(subset='sha256', keep='first')
      .reset_index(drop=True)
)

duplicates_removed = before - len(df)

if duplicates_removed > 0:
    print(f"Removed {duplicates_removed} duplicate sha256 records")


# 4. CHECK CLASS DISTRIBUTION


print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

print("\nNumber of samples:")
print(df['label'].value_counts())

print("\nClass proportions:")
print(
    df['label']
    .value_counts(normalize=True)
    .round(3)
)



# 5. EXTRACT STRUCTURAL NUMERIC FEATURES


def extract_structural_features(sig: str) -> dict:

    max_depth = 0
    current_depth = 0

    # Calculate maximum DOM depth
    for char in sig:

        if char == '(':
            current_depth += 1
            max_depth = max(max_depth, current_depth)

        elif char == ')':
            current_depth = max(0, current_depth - 1)

    # Extract HTML tag names
    tags = re.findall(r'[a-zA-Z][a-zA-Z0-9]*', sig.lower())

    tag_counts = Counter(tags)

    form_count = tag_counts.get("form", 0)
    input_count = tag_counts.get("input", 0)

    return {

        "signature_length": len(sig),

        "max_depth": max_depth,

        "total_tags": len(tags),

        "unique_tags_count": len(tag_counts),

        "form_count": form_count,

        "input_count": input_count,

        "button_count": tag_counts.get("button", 0),

        "div_count": tag_counts.get("div", 0),

        "script_count": tag_counts.get("script", 0),

        "a_count": tag_counts.get("a", 0),

        "img_count": tag_counts.get("img", 0),

        "input_per_form_ratio":
            input_count / form_count
            if form_count > 0 else 0.0
    }


# Apply feature extraction
df_struct = pd.DataFrame(
    [
        extract_structural_features(sig)
        for sig in df['html_signature']
    ]
)

print("\n" + "=" * 60)
print("STRUCTURAL FEATURES")
print("=" * 60)

print(f"Structural feature shape: {df_struct.shape}")

print("\nFirst five rows:")
print(df_struct.head())


# 6. DEFINE X AND y
#

# Features
X_struct = df_struct

# Target
y = df['label']

# Original signatures
signatures = df['html_signature']



# 7. TRAIN / TEST SPLIT
#
# IMPORTANT:
# We split BEFORE fitting the CountVectorizer.
# This prevents information from the test set influencing
# the vocabulary learned from the training set.

X_struct_train, X_struct_test, y_train, y_test, sig_train, sig_test = (
    train_test_split(
        X_struct,
        y,
        signatures,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(f"Training samples: {len(X_struct_train)}")
print(f"Testing samples:  {len(X_struct_test)}")

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).round(3))



# 8. CLEAN HTML SIGNATURES FOR N-GRAM EXTRACTION

def clean_signature(sig):
    """
    Convert the DOM signature into a simple sequence of
    HTML tag tokens.
    """

    tags = re.findall(
        r'[a-zA-Z][a-zA-Z0-9]*',
        sig.lower()
    )

    return ' '.join(tags)


clean_train = sig_train.apply(clean_signature)
clean_test = sig_test.apply(clean_signature)


# ============================================================
# 9. CREATE N-GRAM FEATURES
# ============================================================
#
# The vectorizer is FIT ONLY on the training data.
# The test data is only TRANSFORMED.

vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    token_pattern=r'\b[a-zA-Z][a-zA-Z0-9]*\b',
    min_df=2
)

# Learn vocabulary from training data
X_ngram_train = vectorizer.fit_transform(clean_train)

# Apply the SAME vocabulary to test data
X_ngram_test = vectorizer.transform(clean_test)


print("\n" + "=" * 60)
print("N-GRAM FEATURES")
print("=" * 60)

print(f"Training n-gram shape: {X_ngram_train.shape}")
print(f"Testing n-gram shape:  {X_ngram_test.shape}")

print(
    f"Number of learned n-grams: "
    f"{len(vectorizer.get_feature_names_out())}"
)


# 10. SAVE THE VECTORIZER


with open('ngram_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("\nN-gram vectorizer saved as: ngram_vectorizer.pkl")


# 11. SCALE STRUCTURAL FEATURES
#
# Scaling is useful for models such as Logistic Regression
# and Linear SVM.
#
# We fit the scaler ONLY on training data.
#
scaler = StandardScaler()

X_struct_train_scaled = scaler.fit_transform(
    X_struct_train
)

X_struct_test_scaled = scaler.transform(
    X_struct_test
)


# Save scaler for future test/deployment use
with open('structural_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Structural feature scaler saved as: structural_scaler.pkl")



# 12. CONVERT STRUCTURAL FEATURES TO SPARSE MATRICES

X_struct_train_sparse = csr_matrix(
    X_struct_train_scaled
)

X_struct_test_sparse = csr_matrix(
    X_struct_test_scaled
)


# 13. COMBINE STRUCTURAL + N-GRAM FEATURES


X_train = hstack(
    [
        X_struct_train_sparse,
        X_ngram_train
    ],
    format='csr'
)

X_test = hstack(
    [
        X_struct_test_sparse,
        X_ngram_test
    ],
    format='csr'
)


print("\n" + "=" * 60)
print("FINAL MODEL INPUT")
print("=" * 60)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

# 14. STRUCTURAL FEATURE STATISTICS
print("\n" + "=" * 60)
print("STRUCTURAL FEATURE STATISTICS")
print("=" * 60)

struct_cols = [
    'max_depth',
    'total_tags',
    'form_count',
    'input_count',
    'button_count',
    'script_count'
]

print(
    df_struct[struct_cols]
    .describe()
    .round(2)
)
# 15. CLASS-WISE STRUCTURAL COMPARISON
print("\n" + "=" * 60)
print("CLASS-WISE STRUCTURAL FEATURE MEANS")
print("=" * 60)

comparison_df = pd.concat(
    [
        df['label'].reset_index(drop=True),
        df_struct[struct_cols].reset_index(drop=True)
    ],
    axis=1
)

print(
    comparison_df
    .groupby('label')
    .mean()
    .round(2)
)
# 16. TOP N-GRAM PATTERNS
print("\n" + "=" * 60)
print("TOP 20 N-GRAM PATTERNS IN TRAINING DATA")
print("=" * 60)

ngram_names = vectorizer.get_feature_names_out()

ngram_sums = pd.Series(
    X_ngram_train.sum(axis=0).A1,
    index=ngram_names
)

top_ngrams = (
    ngram_sums
    .sort_values(ascending=False)
    .head(20)
)

for index, (ngram_name, count) in enumerate(
    top_ngrams.items(),
    start=1
):

    print(
        f"{index:2d}. [{ngram_name}] "
        f"appeared {int(count)} times"
    )
# 17. FINAL SUMMARY
print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)

print(f"Total samples: {len(df)}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Total model features: {X_train.shape[1]}")

print("\nReady for machine-learning models.")

Total records loaded: 1183
File 1 records: 502
File 2 records: 681
Found 2 missing html_signature values → replacing with empty strings

CLASS DISTRIBUTION

Number of samples:
label
NO_FORM                 681
LOGIN_FORM_MALICIOUS    502
Name: count, dtype: int64

Class proportions:
label
NO_FORM                 0.576
LOGIN_FORM_MALICIOUS    0.424
Name: proportion, dtype: float64

STRUCTURAL FEATURES
Structural feature shape: (1183, 12)

First five rows:
   signature_length  max_depth  total_tags  unique_tags_count  form_count  \
0               414          9          66                 18           1   
1              3607         13         823                 22           2   
2               392          9          74                 11           0   
3              8304         16        1608                 17           1   
4              1231         11         282                 22           1   

   input_count  button_count  div_count  script_count  a_count  img_count  \
0

In [35]:
!git branch --show-current


feature/ml-model


In [36]:
!git status

On branch feature/ml-model
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

nothing added to commit but untracked files present (use "git add" to track)


In [37]:
!git rev-parse --show-toplevel


/content/html-login-form-detector


In [38]:
!pwd


/content/html-login-form-detector


In [39]:
!find /content -name "*.ipynb"


/content/html-login-form-detector/notebooks/01_data_exploration.ipynb


In [40]:
!git status --short

?? .gitignore


In [41]:
!git diff -- notebooks/01_data_exploration.ipynb


In [20]:
!git status

On branch feature/ml-model
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	ngram_vectorizer.pkl
	structural_scaler.pkl

nothing added to commit but untracked files present (use "git add" to track)


In [21]:
!echo "*.pkl" >> .gitignore


In [22]:
!cat .gitignore

*.pkl


In [26]:
!git add .gitignore

In [27]:
!git push -u origin feature/ml-model


fatal: could not read Username for 'https://github.com': No such device or address


In [28]:
!pwd


/content/html-login-form-detector


In [29]:
!git status


On branch feature/ml-model
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .gitignore



In [30]:
!git restore --staged .gitignore


In [31]:
!git status


On branch feature/ml-model
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

nothing added to commit but untracked files present (use "git add" to track)


In [32]:
!ls -la


total 124
drwxr-xr-x 6 root root  4096 Aug 26 08:57 .
drwxr-xr-x 1 root root  4096 Aug 26 08:32 ..
drwxr-xr-x 3 root root  4096 Aug 26 08:32 data
drwxr-xr-x 8 root root  4096 Aug 26 09:04 .git
-rw-r--r-- 1 root root     6 Aug 26 08:57 .gitignore
-rw-r--r-- 1 root root 94143 Aug 26 08:53 ngram_vectorizer.pkl
drwxr-xr-x 2 root root  4096 Aug 26 08:32 notebooks
drwxr-xr-x 2 root root  4096 Aug 26 08:32 src
-rw-r--r-- 1 root root  1003 Aug 26 08:53 structural_scaler.pkl


In [33]:
!git log --oneline -5


2940e10 (HEAD -> feature/ml-model, origin/main, origin/HEAD, main) Colab을 통해 생성됨
161e2dc Created using Colab
bab8978 Create processed
0731c47 Setup initial structure
d5cedc8 Added raw data files
